# Deep-BSDE 求解高维热方程（对应书中 §7.4）

$$\partial_t u + \Delta u = 0,\quad u(T,x)=\|x\|^2,\qquad
u(t,x)=\|x\|^2+2d\,(T-t)$$

$$X_{n+1}=X_n+\sqrt{2}\,\Delta W_n,\qquad
Y_{n+1}=Y_n+Z_\theta(X_n,t_n)\cdot\Delta W_n,\qquad
\min\;\mathbb{E}\,|Y_N-\|X_N\|^2|^2$$

In [1]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

In [2]:
d, T, N = 10, 0.3, 20
dt = T / N
x0 = torch.zeros(d)

z_net = nn.Sequential(
    nn.Linear(d + 1, 128), nn.SiLU(),
    nn.Linear(128, 128), nn.SiLU(),
    nn.Linear(128, d),
)
y0 = nn.Parameter(torch.tensor(0.0))

opt = torch.optim.Adam(list(z_net.parameters()) + [y0], lr=5e-3)

for epoch in range(3001):
    B = 512
    X = x0.expand(B, d).clone()
    Y = y0.expand(B)
    for n in range(N):
        t = torch.full((B, 1), n * dt)
        dW = (dt ** 0.5) * torch.randn(B, d)
        Z = z_net(torch.cat([X, t], dim=1))
        Y = Y + (Z * dW).sum(dim=1)
        X = X + (2 ** 0.5) * dW
    loss = ((Y - (X ** 2).sum(dim=1)) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 500 == 0:
        print(f"epoch {epoch:5d}   loss = {loss.item():.4f}   Y0 = {y0.item():.4f}")

epoch     0   loss = 43.4225   Y0 = 0.0050


epoch   500   loss = 14.7768   Y0 = 2.2545


epoch  1000   loss = 4.8873   Y0 = 3.9489


epoch  1500   loss = 1.2756   Y0 = 5.0731


epoch  2000   loss = 0.4688   Y0 = 5.6795


epoch  2500   loss = 0.3601   Y0 = 5.9245


epoch  3000   loss = 0.3520   Y0 = 5.9887


In [3]:
u_exact = (x0 ** 2).sum().item() + 2 * d * T
print(f"Deep-BSDE:  u(0,x0) = {y0.item():.4f}")
print(f"exact:      u(0,x0) = {u_exact:.4f}")
print(f"relative error: {abs(y0.item() - u_exact) / u_exact:.2e}")

Deep-BSDE:  u(0,x0) = 5.9887
exact:      u(0,x0) = 6.0000
relative error: 1.88e-03
